# Project 18 — BROKEN notebook (debugging exercise)

Seeded bugs centred on the state-space pitfall: **process vs observation noise confounding**. With vague priors the level either overfits the noise or oversmooths it. Run it, read the diagnostics, find each bug, fix it. Answer key: `BROKEN_BUGS.md` (don't peek first).

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import pytensor.tensor as pt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate()
y = data['y']; T = data['t']

### Model — vague variance priors and a centred random walk.

Watch the $(\sigma_\text{lvl}, \sigma_\text{obs})$ pair plot and the divergences.

In [ ]:
# BUG 1: both variance priors are vague (HalfNormal sigma=10) -> the two
#         noises trade off freely; their posteriors blow up / anti-correlate.
# BUG 2: CENTRED random walk via pm.GaussianRandomWalk -> funnel geometry,
#         divergences when sigma_level is small.
with pm.Model() as model:
    sigma_level = pm.HalfNormal('sigma_level', sigma=10.0)
    sigma_obs = pm.HalfNormal('sigma_obs', sigma=10.0)
    level = pm.GaussianRandomWalk('level', sigma=sigma_level, shape=T,
                                  init_dist=pm.Normal.dist(y.mean(), 5.0))
    pm.Normal('y_obs', mu=level, sigma=sigma_obs, observed=y)
    idata = pm.sample(draws=500, tune=500, chains=2, cores=1,
                      target_accept=0.9, random_seed=RNG, progressbar=False)

In [ ]:
print(az.summary(idata, var_names=['sigma_level','sigma_obs']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))

### The smoking gun — BUG 3: the two variances anti-correlate.

With vague priors the pair plot of $(\sigma_\text{lvl}, \sigma_\text{obs})$ shows a strong negative-correlation banana: the model cannot decide whether the wiggle is drift or noise. Symptom in the fit: the level either chases every point (overfit) or flattens out (oversmooth). Fixes: (1) informative priors like `HalfNormal(0.5)` / `HalfNormal(1.0)`; (2) the **non-centred** random walk from `model.py`.

In [ ]:
az.plot_pair(idata, var_names=['sigma_level','sigma_obs'], kind='scatter',
             scatter_kwargs={'alpha':0.2}); plt.tight_layout()